In [4]:
#导入包和数据集
import torch
from torch.utils.data import DataLoader#加载数据集
from torchvision import transforms#数据预处理
from torchvision import datasets#pytorch自己提供的数据集
import torch.nn.functional as F#激活函数
import torch.optim as optim#优化器


In [9]:
batch_size=64
# 设置 download=False，从本地加载
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    transform=transforms.ToTensor(),
    download=False  # 不从网络下载，只从本地加载
)
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    transform=transforms.ToTensor(),
    download=False  # 不从网络下载，只从本地加载
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)#测试数据集不打乱
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [11]:
#设计卷积层模型
#定义模型类
class CNNModel(torch.nn.Module):#继承标准父类
    def __init__(self):#初始化
        super(CNNModel, self).__init__()#调用父类初始化方法
        #定义卷积层，数据集的通道数是1，输出通道是10，定义卷积核大小为5×5，kernel_size=5
        self.conv1=torch.nn.Conv2d(1, 10, kernel_size=5)
        #再定义第二层卷积层，输入通道数是10，输出通道数是20，卷积核大小也是5×5
        self.conv2=torch.nn.Conv2d(10, 20, kernel_size=5)
        #定义池化层，池化核大小为2×2，kernel_size=2,取这个区间的最大值
        self.pool=torch.nn.MaxPool2d(kernel_size=2)
        #定义全连接层，输入特征数是320，输出特征数是10,这是由卷积层和池化层的输出特征数决定的
        self.fc=torch.nn.Linear(320, 10)
    def forward(self, x):#前向传播方法
        #定义好隐藏层结构，卷积层1，激活函数ReLU，池化层1，卷积层2，激活函数ReLU，池化层2，全连接层
        #flatten数据维数
        batch_size=x.size(0)#获取批次大小,因为x的形状是[batch_size, channels, height, width]
        x=F.relu(self.pool(self.conv1(x)))#卷积层1，池化层,激活函数ReLU
        x=F.relu(self.pool(self.conv2(x)))#卷积层2，池化层,激活函数ReLU
        x=x.view(batch_size, -1)#展平数据，变成二维，第一维是批次大小，第二维是特征数,进行全连接层之前要展平数据
        x=self.fc(x)#全连接层 full connection layer
        return x

In [12]:
#实例化模型
model=CNNModel()
#把计算迁移到GPU上
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")#如果有GPU就用GPU，否则用CPU.cuda:0表示第一个GPU设备
model.to(device)#把模型迁移到设备上
#定义损失函数和优化器
criterion=torch.nn.CrossEntropyLoss()#交叉熵损失函数，适用于多分类问题
optimizer=optim.SGD(model.parameters(), lr=0.1, momentum=0.5)#随机梯度下降优化器，学习率为0.1，动量为0.5

In [13]:
#训练迭代
def train(epoch):
    running_loss=0.0#累计损失
    for batch_idx,data in enumerate(train_loader,0):
        inputs,labels=data#因为train_loader每次返回一个批次的数据，包含输入和标签，所以这里解包成inputs和labels(是一个元组形式)
        inputs,labels=inputs.to(device),labels.to(device)#把输入和标签迁移到设备上,加速训练
        #优化器清零
        optimizer.zero_grad()#清零梯度，准备计算新的梯度
        #前向传播
        outputs=model(inputs)#把输入数据传入模型，得到输出
        #计算损失
        loss=criterion(outputs,labels)#计算输出和标签之间的损失
        #反向传播
        loss.backward()#反向传播，计算梯度
        #优化器更新参数
        optimizer.step()#更新参数，进行梯度下降
        #累计损失
        running_loss+=loss.item()#把当前批次的损失添加到累计损失中
        #每300个批次打印一次平均损失
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 2000))
            running_loss = 0.0

In [15]:
def test():#定义测试函数
    correct=0
    total=0
    with torch.no_grad():#测试时不需要计算梯度，节省内存和计算资源
        for data in test_loader:
            input,labels=data
            inputs,labels=input.to(device),labels.to(device)#把输入和标签迁移到设备上
            outputs=model(inputs)#把输入数据传入模型，得到输出
            _,predicted=torch.max(outputs.data,dim=1)#获取输出中最大值的索引，作为预测结果
            total+=labels.size(0)#统计总的样本数
            correct+=(predicted==labels).sum().item()#统计预测正确的样本数
    print('测试准确率: %.2f %%' % (100 * correct / total))#计算并打印测试准确率




In [16]:
if __name__ == '__main__':#主函数入口
    for epoch in range(10):#训练10个epoch
        train(epoch)
        if epoch %10==9:
            test()

[1,   300] loss: 0.057
[1,   600] loss: 0.017
[1,   900] loss: 0.013
[2,   300] loss: 0.011
[2,   600] loss: 0.010
[2,   900] loss: 0.009
[3,   300] loss: 0.007
[3,   600] loss: 0.008
[3,   900] loss: 0.007
[4,   300] loss: 0.006
[4,   600] loss: 0.006
[4,   900] loss: 0.007
[5,   300] loss: 0.006
[5,   600] loss: 0.005
[5,   900] loss: 0.006
[6,   300] loss: 0.004
[6,   600] loss: 0.005
[6,   900] loss: 0.005
[7,   300] loss: 0.004
[7,   600] loss: 0.005
[7,   900] loss: 0.004
[8,   300] loss: 0.004
[8,   600] loss: 0.003
[8,   900] loss: 0.004
[9,   300] loss: 0.003
[9,   600] loss: 0.003
[9,   900] loss: 0.003
[10,   300] loss: 0.003
[10,   600] loss: 0.003
[10,   900] loss: 0.004
测试准确率: 98.99 %
